# Which aircraft mix covers the service plan?

An independent airline capacity-planning study by Abdulaziz Aldoseri.

An aircraft can provide seats on several routes, but only for a limited number of hours. A service plan also needs sufficient departures, so simply choosing the largest aircraft does not answer the entire sizing question. This study chooses **integer aircraft counts fixed across a year**, together with **integer paired route/type round trips in each month**.

The setting is an eight-route subset of Denver regional service, using SkyWest operating-carrier records in the U.S. DOT/BTS **T-100 Domestic Segment data for 2025**. Three aircraft types are represented: BTS 629, 631 and 673. Reported seats and ramp minutes provide conservative annual planning coefficients. The targets use recorded **carried passengers**, not latent or unmet demand.

The visitor sets a passenger multiplier, a minimum frequency fraction and a daily aircraft operating envelope. The model first minimizes the fleet, then unnecessary round trips and aircraft minutes, while covering each route/month target and each type/month resource constraint. An annual historical-type-share heuristic provides a feasible simple comparison. All twelve months are known: **this is a retrospective planning envelope, not a forecast or timetable, procurement advice, actual airline fleet count or achieved savings**. Remaining ties mean the model displays one minimum-fleet composition, not a unique answer.

Source: U.S. Department of Transportation, Bureau of Transportation Statistics, [T-100 Domestic Segment (U.S. Carriers)](https://www.transtats.bts.gov/DL_SelectFields.aspx?QO_fu146_anzr=Nv4+Pn44vr45&gnoyr_VQ=FIM), 2025; accessed 14 September 2026. Independent analysis; no airline or government endorsement. Source-data reuse is documented in `source_manifest.json`, separately from the MIT licence for original study code.


## 1. Verify the reproduction package

**Local:** extract the study ZIP and open this notebook inside it using Python 3.12 with the pinned packages in `requirements.txt`. Local setup does not install packages or download data.

**Optional Google Colab:** upload this notebook and run setup. When prompted, upload the study reproduction ZIP. The embedded helper validates archive paths, declared sizes and hashes before importing project code; Colab then installs the pinned requirements. No Drive mount is required. Hosted Colab has not been verified; local notebook execution is the release check.

The ZIP contains a 3,956-row original-value BTS carrier/hub subset, frozen model inputs, evaluated outputs and code. Hashes detect corrupted or mismatched content, not publisher identity. Obtain the ZIP from the study's download link and compare the published archive checksum where available. Notebook edits are permitted after strict extraction while modules, data and dependencies remain checked.


In [ ]:
"""Standard-library ZIP validation for the airline fleet reproduction notebook.

Checksums detect corruption and manifest inconsistency, not publisher identity.
Obtain the archive from the study's own download link and compare its published
archive checksum when available. No archive code is executed by this helper.
"""
from __future__ import annotations

from hashlib import sha256
from io import BytesIO
import json
from pathlib import Path, PurePosixPath
import re
import stat
from zipfile import ZipFile

MAX_ARCHIVE_BYTES = 100 * 1024 * 1024
MAX_TOTAL_BYTES = 256 * 1024 * 1024
MAX_FILE_BYTES = 64 * 1024 * 1024
MAX_FILES = 2000
MANIFEST_NAME = "package_manifest.json"


def safe_name(name: str) -> str:
    if not isinstance(name, str) or not name or "\\" in name or ":" in name or "\x00" in name:
        raise ValueError("Invalid archive path")
    path = PurePosixPath(name)
    if path.is_absolute() or any(part in {"", ".", ".."} for part in name.split("/")):
        raise ValueError("Archive path must be relative without traversal")
    if any(part.casefold() in {"private", "private_audit", ".git", ".env", ".deps"} for part in path.parts):
        raise ValueError("Private or repository-internal material is not allowed in the public package")
    if path.suffix.casefold() in {".xlsx", ".xls"} or "journeydataextract" in path.name.casefold():
        raise ValueError("The public package must not contain raw source workbooks or journey CSVs")
    return path.as_posix()


def manifest_entries(raw: bytes) -> dict:
    if len(raw) > 1024 * 1024:
        raise ValueError("Manifest exceeds size limit")
    value = json.loads(raw.decode("utf-8"))
    if not isinstance(value, dict) or not isinstance(value.get("files"), list):
        raise ValueError("Manifest must contain a files array")
    if not 1 <= len(value["files"]) <= MAX_FILES:
        raise ValueError("Invalid manifest file count")
    records, folded = {}, set()
    for item in value["files"]:
        if not isinstance(item, dict):
            raise ValueError("Invalid manifest record")
        name = safe_name(item.get("path"))
        size, digest = item.get("bytes"), item.get("sha256")
        if name == MANIFEST_NAME or name.casefold() in folded:
            raise ValueError("Duplicate, case-colliding or self-referencing manifest path")
        if isinstance(size, bool) or not isinstance(size, int) or not 0 <= size <= MAX_FILE_BYTES:
            raise ValueError("Invalid manifest size")
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
            raise ValueError("Invalid SHA-256 digest")
        records[name] = {"bytes": size, "sha256": digest}
        folded.add(name.casefold())
    if sum(record["bytes"] for record in records.values()) > MAX_TOTAL_BYTES:
        raise ValueError("Manifest total exceeds size limit")
    return records


def verify_package(directory: str | Path, allow_notebook_edits: bool = False) -> dict:
    """Verify declared files; optional working-notebook edits do not exempt code/data.

    Extraction always uses strict verification. At notebook runtime, Jupyter
    saves outputs and edited parameters into .ipynb files, so those working
    documents may differ while all executable modules/data/requirements remain
    hash-checked. Notebook paths still must exist and be regular bounded files.
    """
    root = Path(directory).resolve()
    manifest = root / MANIFEST_NAME
    if manifest.is_symlink() or not manifest.is_file():
        raise ValueError("Package manifest is missing or is a symlink")
    entries = manifest_entries(manifest.read_bytes())
    for name, item in entries.items():
        target = root / name
        if any(part.is_symlink() for part in [target, *target.parents] if part != root.parent):
            raise ValueError("Symlinks are not permitted in a package")
        if not target.resolve().is_relative_to(root) or not target.is_file():
            raise ValueError("Missing or unsafe package file")
        if allow_notebook_edits and target.suffix.casefold() == ".ipynb":
            if target.stat().st_size > MAX_FILE_BYTES:
                raise ValueError("Working notebook exceeds size limit")
            continue
        if target.stat().st_size != item["bytes"] or sha256(target.read_bytes()).hexdigest() != item["sha256"]:
            raise ValueError("Package checksum or size mismatch: " + name)
    return entries


def safe_extract_package(archive: bytes | str | Path, destination: str | Path) -> Path:
    """Verify all members before writing to a new/empty destination directory."""
    if isinstance(archive, bytes):
        raw = archive
    else:
        path = Path(archive)
        if path.stat().st_size > MAX_ARCHIVE_BYTES:
            raise ValueError("Archive exceeds compressed size limit")
        raw = path.read_bytes()
    if len(raw) > MAX_ARCHIVE_BYTES:
        raise ValueError("Archive exceeds compressed size limit")
    output = Path(destination)
    if output.is_symlink():
        raise ValueError("Destination must not be a symlink")
    output = output.resolve()
    if output.exists() and (not output.is_dir() or any(output.iterdir())):
        raise ValueError("Choose a new or empty extraction directory")
    with ZipFile(BytesIO(raw)) as archive_zip:
        infos = archive_zip.infolist()
        if len(infos) > MAX_FILES + 1:
            raise ValueError("Archive has too many entries")
        names, folded, total = {}, set(), 0
        for info in infos:
            name = safe_name(info.filename.rstrip("/") if info.is_dir() else info.filename)
            if name.casefold() in folded:
                raise ValueError("Duplicate or case-colliding archive member")
            folded.add(name.casefold())
            kind = stat.S_IFMT(info.external_attr >> 16)
            if kind not in {0, stat.S_IFREG, stat.S_IFDIR} or (kind == stat.S_IFDIR and not info.is_dir()):
                raise ValueError("Archive contains a nonregular entry")
            if info.flag_bits & 1:
                raise ValueError("Encrypted archive members are not supported")
            if info.is_dir():
                continue
            if not 0 <= info.file_size <= MAX_FILE_BYTES:
                raise ValueError("Archive member exceeds size limit")
            total += info.file_size
            names[name] = info
        if total > MAX_TOTAL_BYTES:
            raise ValueError("Archive exceeds expanded size limit")
        file_names = {name.casefold() for name in names}
        prefixes = {}
        for name in names:
            parts = PurePosixPath(name).parts
            for end in range(1, len(parts) + 1):
                prefix = "/".join(parts[:end])
                folded_prefix = prefix.casefold()
                if folded_prefix in prefixes and prefixes[folded_prefix] != prefix:
                    raise ValueError("Inconsistent case in archive path components")
                prefixes[folded_prefix] = prefix
                if end < len(parts) and folded_prefix in file_names:
                    raise ValueError("Archive file conflicts with a required directory")
        if MANIFEST_NAME not in names:
            raise ValueError("Archive root must contain package_manifest.json")
        if names[MANIFEST_NAME].file_size > 1024 * 1024:
            raise ValueError("Manifest exceeds size limit")
        manifest = archive_zip.read(names[MANIFEST_NAME])
        entries = manifest_entries(manifest)
        if set(names) != set(entries) | {MANIFEST_NAME}:
            raise ValueError("Every archive file must appear exactly once in the manifest")
        verified = {MANIFEST_NAME: manifest}
        for name, item in entries.items():
            info = names[name]
            if info.file_size != item["bytes"]:
                raise ValueError("Member size disagrees with manifest")
            with archive_zip.open(info) as member:
                contents = member.read(MAX_FILE_BYTES + 1)
            if len(contents) != item["bytes"] or sha256(contents).hexdigest() != item["sha256"]:
                raise ValueError("Member checksum disagrees with manifest: " + name)
            verified[name] = contents
    # The manifest is validated before any data or executable source is written.
    output.mkdir(parents=True, exist_ok=True)
    for name, contents in verified.items():
        target = output / name
        if not target.resolve().is_relative_to(output):
            raise ValueError("Unsafe extraction target")
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("xb") as handle:
            handle.write(contents)
    verify_package(output)
    return output


In [ ]:
import importlib.util, subprocess, sys
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except (ImportError, ModuleNotFoundError):
    IN_COLAB = False
PROJECT = Path.cwd().resolve()
if not (PROJECT/'fleet_model.py').is_file():
    extracted = PROJECT/'airline-fleet-reproduction'
    if (extracted/'fleet_model.py').is_file():
        verify_package(extracted, allow_notebook_edits=True)
        PROJECT = extracted
    elif IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Upload exactly one study reproduction ZIP.')
        filename, contents = next(iter(uploaded.items()))
        if not filename.lower().endswith('.zip'):
            raise ValueError('Select the study reproduction ZIP.')
        PROJECT = safe_extract_package(contents, extracted)
    else:
        raise FileNotFoundError('Open the notebook inside the extracted study folder.')
if not (PROJECT/'package_manifest.json').is_file():
    raise ValueError('The reproduction package manifest is required before importing study code.')
verify_package(PROJECT, allow_notebook_edits=True)
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(PROJECT/'requirements.txt')])
sys.path.insert(0,str(PROJECT))
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value,'to_string') else value)
from prepare_data import prepare
from pipeline import verify_freeze, run
from fleet_model import cap_feasibility
freeze=verify_freeze()
protocol=json.loads((PROJECT/'PROTOCOL.json').read_text())
print('Verified frozen inputs and model. All targets are known historical 2025 observations.')


## 2. Rebuild the observed-data inputs

Preparation rejects changed source bytes, preserves the original source subset, excludes the 75 other-type and 36 DEN–DEN self-route records, and recomputes all route eligibility and coefficients. The eight selected routes cover **23.26% of the retained SkyWest/Denver traffic**, not the complete network. Targets use the larger directional carried-passenger and performed-departure counts; a paired round trip supplies one departure and the stated seat coefficient in each direction.

Seats are the floor of the smaller directional annual weighted average. Paired block minutes sum the two directional annual means, each rounded up. The scenario adds **30 minutes per leg** and retains **10% of the daily envelope as unavailable/reserve**. These operating allowances are assumptions, not measured airline values. The minute constraint does not reconstruct aircraft circulation, flight times, staffing, maintenance, runway/range feasibility, slots or gates.


In [ ]:
data,audit=prepare()
verify_freeze()  # deterministic preparation must reproduce the frozen input bytes
display(pd.DataFrame(audit['selected_routes']))
display(pd.DataFrame([{'route':r['destination'],'type':t,**c} for r in data['routes'] for t,c in r['coefficients'].items()]))
print(f"Selected traffic share: {audit['selected_share_of_carrier_hub_passengers']:.2%}")


## 3. Reproduce all 27 evaluated scenarios

The grid crosses passenger targets of 80%, 100% and 120% of carried traffic; frequency floors of 50%, 75% and 100% of performed departures; and 6, 8 or 10 daily aircraft hours. Each result sizes a fleet shared across all twelve months. Every lexicographic solver stage must finish optimally with zero reported gap, then pass exact integer service and resource checks.

The comparison preserves annual eligible type shares using **largest-remainder apportionment of an exact total**. It searches for enough round trips to meet the same targets, then sizes a sufficient fleet for its busiest month by type. It is a feasible heuristic, not the actual airline fleet.

The cell writes to a fresh output directory and refuses to overwrite completed results. To perform another full rerun, change `RUN_DIRECTORY` to a new directory name. The input/code freeze is checked again before final output files are written.


In [ ]:
RUN_DIRECTORY='notebook_reproduction'
index=run(RUN_DIRECTORY)
summary=pd.read_csv(PROJECT/RUN_DIRECTORY/'scenario_summary.csv')
display(summary)
assert len(index['scenarios'])==27
assert all(stage['relative_gap']==0 for case in index['scenarios'] for stage in case['optimized']['solver'])


## 4. Inspect a service and resource choice

Change only the evaluated scenario ID and month below. These controls select an evaluated case; they do not interpolate or perform a fresh arbitrary solve. The default was fixed before evaluation. Fleet counts serve the full year even when one month's assignments are displayed.


In [ ]:
SCENARIO_ID='p100-f75-h8'
MONTH=1
case=next(case for case in index['scenarios'] if case['id']==SCENARIO_ID)
routes={r['id']:r['destination'] for r in index['routes']}
fleet_rows=[{'policy':policy,'type':t,'aircraft':n} for policy in ['optimized','baseline'] for t,n in case[policy]['fleet_by_type'].items()]
display(pd.DataFrame(fleet_rows))
display(pd.DataFrame([{**a,'destination':routes[a['route_id']]} for a in case['optimized']['assignments'] if a['month']==MONTH]))
display(pd.DataFrame([r for r in case['optimized']['route_outcomes'] if r['month']==MONTH]))
display(pd.DataFrame([r for r in case['optimized']['utilization'] if r['month']==MONTH]))
print(f"One minimum-fleet solution: {case['optimized']['total_fleet']} aircraft; historical-share policy: {case['baseline']['total_fleet']}.")


## 5. Make a hard limit visible

A cap one aircraft below the proven minimum makes the stated service envelope infeasible. Each policy must be checked separately: a historical-share allocation may exceed a cap that the optimized allocation meets. This is an exact threshold for the chosen evaluated scenario, not an extrapolated capacity curve.


In [ ]:
FLEET_CAP=max(0,case['optimized']['total_fleet']-1)
display(pd.DataFrame([{'policy':policy,'fleet_required':case[policy]['total_fleet'],'cap':FLEET_CAP,'feasible_at_cap':cap_feasibility(case[policy],FLEET_CAP)} for policy in ['optimized','baseline']]))
assert not cap_feasibility(case['optimized'],case['optimized']['total_fleet']-1)
assert cap_feasibility(case['optimized'],case['optimized']['total_fleet'])


## 6. Verify limiting cases and interpret the result

The tests compare small integer instances with exhaustive enumeration, including the two secondary objectives. They also check zero service, single-type cases, changed source rejection, exact-total shares, resource violations and cap feasibility. Artificial numbers appear only in mathematical unit-test fixtures, not as observational airline data.

A lower required aircraft count reflects the model's freely pooled monthly hours and route/type mix. It is **not savings**, evidence that SkyWest could remove aircraft, or a decision to buy a specific airframe. An operational extension would need actual fleet availability, tail circulation, timetable and within-day peaks, crews, maintenance, airport feasibility, contracts and costs. See `METHODS.md`, `DATA_PREPARATION.md`, `PROTOCOL.json` and `RESULTS.md` for the full scope and recorded findings.


In [ ]:
completed=subprocess.run([sys.executable,'-m','unittest','discover','-s',str(PROJECT/'tests'),'-v'],cwd=PROJECT,text=True,capture_output=True)
print(completed.stdout+completed.stderr)
assert completed.returncode==0
